# Pipeline final de análise espectral

Este notebook organiza o fluxo metodológico completo do projeto. Nesta versão, somente as etapas já consolidadas são executadas: **aquisição dos dados**, **pré-processamento espectral**, **teste de normalidade** e **teste de diferenças significativas**. As etapas seguintes permanecem como placeholders para integração futura.

Configuração estatística consolidada: espectro normalizado por SNV, leituras individuais, turno da manhã e parâmetros definidos nos módulos originais.

In [1]:
from __future__ import annotations

import io
import os
import sys
import time
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path
from unittest.mock import patch

import pandas as pd
from IPython.display import display


def localizar_raiz(inicio: Path | None = None) -> Path:
    inicio = (inicio or Path.cwd()).resolve()
    for candidato in (inicio, *inicio.parents):
        if (candidato / "dataset").is_dir() and (candidato / "preprocessamento_espectral").is_dir():
            return candidato
    raise FileNotFoundError("Não foi possível localizar a raiz de EstresseHidricoFinal.")


ROOT = localizar_raiz()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dataset import limpar_metadados
from preprocessamento_espectral import normalizacao, preprocessamento, suavizacao
from testeDeNormalidade import shapiro_normalidade
from testeDiferencaSignificativa import diferencas_significativas

LOGS: dict[str, str] = {}


def executar(etapa: str, funcao, *argumentos: str) -> None:
    captura = io.StringIO()
    inicio = time.perf_counter()
    argv = [f"{funcao.__module__}.py", *argumentos]
    with patch.object(sys, "argv", argv), redirect_stdout(captura), redirect_stderr(captura):
        funcao()
    LOGS[etapa] = captura.getvalue()
    print(f"✓ {etapa} concluída em {time.perf_counter() - inicio:.1f} s")


print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: C:\Vitor\SENAI\EstressFinal\EstresseHidricoFinal


## 1. Aquisição dos dados espectrais

Carregamento da base unificada e aplicação das correções consolidadas de metadados, sem alterar as leituras espectrais.

In [2]:
executar("Aquisição e limpeza dos dados", limpar_metadados.main)

✓ Aquisição e limpeza dos dados concluída em 9.3 s


In [3]:
dados_adquiridos = pd.read_csv(limpar_metadados.SAIDA, sep=";", decimal=",")
dados_adquiridos.columns = dados_adquiridos.columns.str.strip()
metadados = [coluna for coluna in dados_adquiridos.columns if not coluna.isdigit()]
bandas = sorted(int(coluna) for coluna in dados_adquiridos.columns if coluna.isdigit())

resumo_aquisicao = pd.DataFrame({
    "métrica": [
        "amostras", "metadados", "bandas espectrais", "faixa espectral",
        "valores ausentes", "linhas duplicadas", "genótipos", "condições",
    ],
    "valor": [
        len(dados_adquiridos), len(metadados), len(bandas), f"{bandas[0]}–{bandas[-1]} nm",
        int(dados_adquiridos.isna().sum().sum()), int(dados_adquiridos.duplicated().sum()),
        ", ".join(sorted(dados_adquiridos["genotipo"].dropna().astype(str).unique())),
        ", ".join(sorted(dados_adquiridos["condicao"].dropna().astype(str).unique())),
    ],
})
display(resumo_aquisicao)
del dados_adquiridos

,métrica,valor
0,amostras,1924
1,metadados,6
2,bandas espectrais,2151
3,faixa espectral,350–2500 nm
4,valores ausentes,0
5,linhas duplicadas,0
6,genótipos,"BR16, CD202, EMB48"
7,condições,"IRRIG, NIRRIG"


## 2. Correções e pré-processamento espectral

Execução sequencial de correção das emendas dos detectores, limpeza e interpolação, recorte para 400–2450 nm, suavização Savitzky–Golay e normalização SNV.

In [4]:
executar("Correção, limpeza e recorte espectral", preprocessamento.main)
executar("Suavização Savitzky–Golay", suavizacao.main)
executar("Normalização SNV", normalizacao.main)

✓ Correção, limpeza e recorte espectral concluída em 8.3 s


✓ Suavização Savitzky–Golay concluída em 9.1 s


✓ Normalização SNV concluída em 8.3 s


In [5]:
arquivos_preprocessamento = {
    "corrigido e recortado": preprocessamento.SAIDA,
    "suavizado": suavizacao.SAIDA,
    "normalizado (SNV)": normalizacao.SAIDA,
}

linhas_resumo = []
for estagio, arquivo in arquivos_preprocessamento.items():
    dados_estagio = pd.read_csv(arquivo, sep=";", decimal=",")
    bandas_estagio = sorted(int(c) for c in dados_estagio.columns if c.isdigit())
    linhas_resumo.append({
        "estágio": estagio,
        "amostras": len(dados_estagio),
        "bandas": len(bandas_estagio),
        "faixa_nm": f"{bandas_estagio[0]}–{bandas_estagio[-1]}",
        "ausentes": int(dados_estagio.isna().sum().sum()),
        "arquivo": arquivo.name,
    })
    del dados_estagio

resumo_preprocessamento = pd.DataFrame(linhas_resumo)
display(resumo_preprocessamento)

,estágio,amostras,bandas,faixa_nm,ausentes,arquivo
0,corrigido e recortado,1924,2051,400–2450,0,Unificada13052026_400_2450.csv
1,suavizado,1924,2051,400–2450,0,Unificada13052026_suavizado.csv
2,normalizado (SNV),1924,2051,400–2450,0,Unificada13052026_normalizado.csv


## 3. Teste de normalidade

Shapiro–Wilk por banda sobre o espectro normalizado, com correção FDR de Benjamini–Hochberg e agrupamentos por genótipo, condição e dia. Unidade estatística: leitura individual; turno: manhã.

In [6]:
executar("Teste de normalidade", shapiro_normalidade.main, "normalizado", "leitura")

✓ Teste de normalidade concluída em 32.5 s


In [7]:
normalidade = pd.read_csv(
    shapiro_normalidade.SAIDA_DIR / "normalidade_resumo.csv", sep=";"
)
normalidade_bandas = pd.read_csv(
    shapiro_normalidade.SAIDA_DIR / "normalidade_por_banda.csv", sep=";"
)

resumo_normalidade = (
    normalidade.groupby("agrupamento", as_index=False)
    .agg(
        grupos=("grupo", "size"),
        n_mínimo=("n_amostras", "min"),
        n_máximo=("n_amostras", "max"),
        proporção_normal_FDR_média=("prop_normais_q", "mean"),
        W_mediano=("W_mediano", "median"),
    )
)
resumo_normalidade[["proporção_normal_FDR_média", "W_mediano"]] = (
    resumo_normalidade[["proporção_normal_FDR_média", "W_mediano"]].round(3)
)
display(resumo_normalidade)

bandas_normais_todos_estratos = int(
    normalidade_bandas["prop_estratos_normais_q"].eq(1).sum()
)
print(
    f"Bandas normais após FDR em todos os estratos completos: "
    f"{bandas_normais_todos_estratos} de {len(normalidade_bandas)}."
)

,agrupamento,grupos,n_mínimo,n_máximo,proporção_normal_FDR_média,W_mediano
0,condicao,2,672,676,0.055,0.991
1,dia,7,192,196,0.134,0.979
2,genotipo,3,448,452,0.062,0.986
3,genotipo_condicao_dia,42,32,36,0.321,0.910
4,global,1,1348,1348,0.047,0.992


Bandas normais após FDR em todos os estratos completos: 0 de 2051.


## 4. Teste de diferenças significativas

Para cada dia e banda, a normalidade das seis células do delineamento define a via: ANOVA fatorial para dados normais ou Kruskal–Wallis com pós-hoc de Dunn para dados não normais. Os p-valores recebem correção FDR.

In [8]:
executar("Teste de diferenças significativas", diferencas_significativas.main, "normalizado")

✓ Teste de diferenças significativas concluída em 26.4 s


In [9]:
diferencas = pd.read_csv(
    diferencas_significativas.SAIDA_DIR / "diferencas_resumo.csv", sep=";"
)
colunas_resumo = [
    "dia", "n_amostras", "bandas", "via_anova", "via_kruskal",
    "sig_genotipo", "sig_condicao", "sig_interacao", "sig_omnibus",
    "pares_dunn_sig",
]
display(diferencas[colunas_resumo].fillna("não aplicável"))

,dia,n_amostras,bandas,via_anova,via_kruskal,sig_genotipo,sig_condicao,sig_interacao,sig_omnibus,pares_dunn_sig
0,D02,192,2051,0,2051,616,809,não aplicável,2037,10202
1,D03,192,2051,0,2051,1800,1914,não aplicável,1937,19954
2,D04,192,2051,0,2051,1766,684,não aplicável,2051,15371
3,D05,192,2051,0,2051,2029,2026,não aplicável,2051,22574
4,D06,192,2051,0,2051,183,2032,não aplicável,2051,20459
5,D09,196,2051,0,2051,180,1979,não aplicável,1992,19860
6,D10,192,2051,0,2051,780,634,não aplicável,2051,16124


## 5. Análise temporal das bandas

> **Placeholder — não executado nesta versão.** Integrar a análise de medidas repetidas/blocos temporais após a consolidação final desta etapa.

## 6. Redução de colinearidade

> **Placeholder — não executado nesta versão.** Integrar a redução por correlação de Spearman e seleção de bandas representativas após a consolidação final desta etapa.

## 7. Seleção de variáveis

> **Placeholder — não executado nesta versão.** Integrar VIP, Boruta e seleção final das bandas quando a metodologia estiver consolidada.

## 8. Classificação

> **Placeholder — não executado nesta versão.** Integrar os classificadores e suas métricas somente após a definição final das variáveis de entrada.

## 9. Desenvolvimento de novo Índice Espectral

> **Placeholder — não executado nesta versão.** Integrar a construção e avaliação do novo índice depois da seleção e validação das bandas finais.

## Validação da execução

Verificações mínimas dos contratos entre as etapas já consolidadas.

In [10]:
saidas_obrigatorias = [
    limpar_metadados.SAIDA,
    *arquivos_preprocessamento.values(),
    shapiro_normalidade.SAIDA_DIR / "normalidade_resumo.csv",
    shapiro_normalidade.SAIDA_DIR / "normalidade_por_banda.csv",
    diferencas_significativas.SAIDA_DIR / "diferencas_resumo.csv",
    diferencas_significativas.SAIDA_DIR / "diferencas_por_banda_dia.csv",
]

assert all(arquivo.exists() for arquivo in saidas_obrigatorias), "Há saídas obrigatórias ausentes."
assert set(resumo_preprocessamento["bandas"]) == {2051}, "Quantidade inesperada de bandas."
assert set(resumo_preprocessamento["faixa_nm"]) == {"400–2450"}, "Faixa espectral inesperada."
assert set(normalidade["agrupamento"]) == {
    "global", "genotipo", "condicao", "dia", "genotipo_condicao_dia"
}, "Agrupamentos de normalidade incompletos."
assert set(diferencas["dia"]) == {"D02", "D03", "D04", "D05", "D06", "D09", "D10"}, (
    "Dias ausentes no resumo de diferenças."
)

display(pd.DataFrame({
    "verificação": ["artefatos", "bandas", "faixa espectral", "normalidade", "diferenças"],
    "resultado": ["OK", "2.051", "400–2450 nm", "5 agrupamentos", "7 dias"],
}))

,verificação,resultado
0,artefatos,OK
1,bandas,2.051
2,faixa espectral,400–2450 nm
3,normalidade,5 agrupamentos
4,diferenças,7 dias
